<a href="https://colab.research.google.com/github/Pazidu/Research-Project/blob/main/june22_test1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/drive')

Drive already mounted at /drive; to attempt to forcibly remount, call drive.mount("/drive", force_remount=True).


In [2]:
import os
import shutil
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2S
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras import mixed_precision
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow version:", tf.__version__)
print("GPU:", tf.test.gpu_device_name())

TensorFlow version: 2.20.0
GPU: /device:GPU:0


In [3]:
BASE            = "/content/newdata"
IMG_SRC         = "/drive/MyDrive/Colab Notebooks/newdata"
CHECKPOINT_DIR  = "/drive/MyDrive/checkpoints"
MODEL_SAVE_PATH = "/drive/MyDrive/Colab Notebooks/Models/dermoscopy/efficientnetv2s_dual_branch_run12.keras"


In [4]:
if os.path.exists(BASE):
    shutil.rmtree(BASE)
shutil.copytree(IMG_SRC, BASE)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

In [5]:
mixed_precision.set_global_policy("float32")

BATCH_SIZE       = 8               # <-- CHANGED: reduced from 16 because 456×456
                                   # images are 3x larger in memory
IMAGE_SIZE       = 456             # <-- CHANGED: matches paper's EfficientNet-B5
                                   # input size, captures finer dermoscopic detail
FUSION_LAYER     = "block4c_add"
EPOCHS           = 30             # matches the paper
STEPS_PER_EPOCH  = (7122 + 890) // BATCH_SIZE

CLASS_ORDER      = ["melanoma", "non_melanoma"]

In [6]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.10),
    layers.RandomZoom(0.1),
], name="data_augmentation")

In [7]:
def add_edge_map(image, label):
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)
    gray  = tf.image.rgb_to_grayscale(image)
    sobel = tf.image.sobel_edges(gray)
    edge  = tf.sqrt(tf.reduce_sum(tf.square(sobel), axis=-1))
    edge  = edge / (tf.reduce_max(edge) + 1e-6)
    return (image, edge), label

In [8]:
def make_class_stream(base_path, only_class):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        base_path,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE // 2,
        label_mode="categorical",
        shuffle=True,
        class_names=CLASS_ORDER,
    )
    ds = ds.filter(lambda x, y: tf.equal(tf.argmax(y[0]), only_class))
    ds = ds.repeat()
    ds = ds.map(
        lambda x, y: (data_augmentation(x, training=True), y),
        num_parallel_calls=tf.data.AUTOTUNE,
    )
    return ds


melanoma_stream     = make_class_stream(f"{BASE}/train", only_class=0)
non_melanoma_stream = make_class_stream(f"{BASE}/train", only_class=1)

balanced_ds = tf.data.Dataset.sample_from_datasets(
    [melanoma_stream, non_melanoma_stream],
    weights=[0.5, 0.5],
)
balanced_ds = balanced_ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
balanced_ds = balanced_ds.prefetch(tf.data.AUTOTUNE)

Found 8012 files belonging to 2 classes.
Found 8012 files belonging to 2 classes.


In [9]:
def prepare_eval_dataset(path):
    ds = tf.keras.preprocessing.image_dataset_from_directory(
        path,
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        label_mode="categorical",
        shuffle=False,
        class_names=CLASS_ORDER,
    )
    ds = ds.map(add_edge_map, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.prefetch(tf.data.AUTOTUNE)
    return ds


val_ds  = prepare_eval_dataset(f"{BASE}/valid")
test_ds = prepare_eval_dataset(f"{BASE}/test")

Found 1001 files belonging to 2 classes.
Found 1002 files belonging to 2 classes.


In [10]:
def create_dual_model(steps_per_epoch):
    rgb_input  = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 3), name="rgb_input")
    base_model = EfficientNetV2S(
        include_top=False,
        weights="imagenet",
        input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    )

    for layer in base_model.layers[:-40]:
        layer.trainable = False

    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=base_model.get_layer(FUSION_LAYER).output,
    )
    middle_feature = feature_extractor(rgb_input)

    # --- Edge branch ---
    edge_input = layers.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, 1), name="edge_input")
    x = layers.Conv2D(32,  3, activation="relu", padding="same")(edge_input)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64,  3, activation="relu", padding="same")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, activation="relu", padding="same")(x)
    x = layers.Resizing(middle_feature.shape[1], middle_feature.shape[2])(x)
    x = layers.Conv2D(middle_feature.shape[-1], 1, padding="same")(x)

    # --- Feature fusion ---
    fused = layers.Concatenate()([middle_feature, x])
    fused = layers.Conv2D(256, 3, activation="relu", padding="same")(fused)
    fused = layers.BatchNormalization()(fused)
    fused = layers.GlobalAveragePooling2D()(fused)
    fused = layers.Dense(
        256, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(1e-4),
    )(fused)
    fused   = layers.Dropout(0.5)(fused)
    outputs = layers.Dense(2, activation="softmax")(fused)

    cosine_lr = tf.keras.optimizers.schedules.CosineDecay(
        initial_learning_rate=2e-4,
        decay_steps=EPOCHS * steps_per_epoch,
    )

    model = tf.keras.Model(inputs=[rgb_input, edge_input], outputs=outputs)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(cosine_lr),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
        metrics=[
            "accuracy",
            tf.keras.metrics.AUC(name="auc"),
            tf.keras.metrics.Recall(name="sensitivity"),
            tf.keras.metrics.Recall(name="specificity", class_id=1),
            tf.keras.metrics.Precision(name="precision"),
        ],
    )
    return model

In [11]:
model = create_dual_model(steps_per_epoch=STEPS_PER_EPOCH)
model.summary()

Model: "functional_2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ edge_input          │ (None, 456, 456,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 456, 456,  │        320 │ edge_input[0][0]  │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 228, 228,  │          0 │ conv2d[0][0]      │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 228, 228,  │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 114, 114,  │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 114, 114,  │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rgb_input           │ (None, 456, 456,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ resizing (Resizing) │ (None, 29, 29,    │          0 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ functional_1        │ (None, 29, 29,    │  1,317,880 │ rgb_input[0][0]   │
│ (Functional)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 29, 29,    │     16,512 │ resizing[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 29, 29,    │          0 │ functional_1[0][… │
│ (Concatenate)       │ 256)              │            │ conv2d_3[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 29, 29,    │    590,080 │ concatenate[0][0] │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 29, 29,    │      1,024 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ batch_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 256)       │     65,792 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 256)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 2)         │        514 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,084,474 (7.95 MB)

 Trainable params: 766,082 (2.92 MB)

 Non-trainable params: 1,318,392 (5.03 MB)

In [12]:
checkpoint_best = ModelCheckpoint(
    filepath=f"{CHECKPOINT_DIR}/best_dual_{FUSION_LAYER}_run12.keras",
    monitor="val_auc",
    save_best_only=True,
    verbose=1,
)

early_stopping = EarlyStopping(
    monitor="val_auc",
    patience=10,
    restore_best_weights=True,
    verbose=1,
)

In [13]:
history = model.fit(
    balanced_ds,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_data=val_ds,
    callbacks=[checkpoint_best, early_stopping],
)

Epoch 1/30
1001/1001 ━━━━━━━━━━━━━━━━━━━━ 0s 402ms/step - accuracy: 0.7929 - auc: 0.8819 - loss: 0.4920 - precision: 0.7929 - sensitivity: 0.7929 - specificity: 0.9102
Epoch 1: val_auc improved from None to 0.74269, saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add_run12.keras

Epoch 1: finished saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add_run12.keras
1001/1001 ━━━━━━━━━━━━━━━━━━━━ 490s 460ms/step - accuracy: 0.8109 - auc: 0.8941 - loss: 0.4738 - precision: 0.8109 - sensitivity: 0.8109 - specificity: 0.9265 - val_accuracy: 0.6414 - val_auc: 0.7427 - val_loss: 0.9081 - val_precision: 0.6414 - val_sensitivity: 0.6414 - val_specificity: 0.6090
Epoch 2/30
1001/1001 ━━━━━━━━━━━━━━━━━━━━ 0s 398ms/step - accuracy: 0.8086 - auc: 0.9031 - loss: 0.4572 - precision: 0.8086 - sensitivity: 0.8086 - specificity: 0.9266
Epoch 2: val_auc improved from 0.74269 to 0.90552, saving model to /drive/MyDrive/checkpoints/best_dual_block4c_add_run12.keras

Epoch 2: finished s

In [15]:
print("\n── Test results (run 12: 456×456 image size) ──")
results = model.evaluate(test_ds)
for name, val in zip(model.metrics_names, results):
    print(f"  {name}: {val:.4f}")

model.save(MODEL_SAVE_PATH)


── Test results (run 12: 456×456 image size) ──
126/126 ━━━━━━━━━━━━━━━━━━━━ 46s 366ms/step - accuracy: 0.6856 - auc: 0.7848 - loss: 0.7506 - precision: 0.6856 - sensitivity: 0.6856 - specificity: 0.6596
  loss: 0.7506
  compile_metrics: 0.6856


In [16]:
import os

root_dir = "/drive/MyDrive/Colab Notebooks/newdata"

print(f"\nDataset Structure: {root_dir}\n")

total_images = 0

for root, dirs, files in os.walk(root_dir):
    image_files = [
        f for f in files
        if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.webp'))
    ]

    level = root.replace(root_dir, '').count(os.sep)
    indent = '│   ' * level
    folder_name = os.path.basename(root)

    print(f"{indent}├── {folder_name} ({len(image_files)} images)")

    total_images += len(image_files)

print(f"\nTotal Images: {total_images}")


Dataset Structure: /drive/MyDrive/Colab Notebooks/newdata

├── newdata (0 images)
│   ├── train (0 images)
│   │   ├── melanoma (890 images)
│   │   ├── non_melanoma (7122 images)
│   ├── test (0 images)
│   │   ├── melanoma (112 images)
│   │   ├── non_melanoma (890 images)
│   ├── valid (0 images)
│   │   ├── melanoma (111 images)
│   │   ├── non_melanoma (890 images)

Total Images: 10015
